In [7]:
# Импорты и переменные окружения

import json
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

True

In [8]:
# Golden dataset из 10 эталонных вопросов

GOLDEN = [
    # --- 7 EN in-corpus вопросов ---
    {
        "question": "How does Ridge regression handle multicollinearity?",
        "ground_truth": "Ridge adds an L2 penalty alpha * sum(w_i^2) to the loss, "
                        "which shrinks correlated coefficients toward each other.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "What does the alpha parameter control in Ridge?",
        "ground_truth": "Alpha controls regularization strength; larger alpha means "
                        "stronger penalty and smaller coefficients.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "What is the difference between Lasso and Ridge?",
        "ground_truth": "Lasso uses L1 penalty which can zero out coefficients (feature "
                        "selection); Ridge uses L2 which shrinks but never zeroes.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "What does min_samples_leaf control in a decision tree?",
        "ground_truth": "min_samples_leaf is the minimum number of samples required to be "
                        "at a leaf node; higher values prevent overfitting by limiting depth.",
        "ground_truth_url_keywords": ["tree"],
    },
    {
        "question": "When does a decision tree overfit?",
        "ground_truth": "Trees overfit when grown too deep without min_samples_leaf or "
                        "min_samples_split constraints, memorising training noise.",
        "ground_truth_url_keywords": ["tree"],
    },
    {
        "question": "What is the formula for precision?",
        "ground_truth": "precision = TP / (TP + FP). Fraction of positive predictions that "
                        "are actually positive.",
        "ground_truth_url_keywords": ["model_evaluation"],
    },
    {
        "question": "When is recall more important than precision?",
        "ground_truth": "Recall matters most when missing positives is costly: cancer "
                        "screening, fraud detection, anything where false negatives are "
                        "more harmful than false positives.",
        "ground_truth_url_keywords": ["model_evaluation"],
    },
    # --- 2 RU вопроса (тест мультиязычного retrieval) ---
    {
        "question": "Что такое L2-регуляризация?",
        "ground_truth": "L2-регуляризация добавляет к функции потерь штраф, "
                        "пропорциональный сумме квадратов коэффициентов модели. "
                        "Используется в Ridge.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "Что такое F1-мера?",
        "ground_truth": "F1 — гармоническое среднее precision и recall, "
                        "F1 = 2 * precision * recall / (precision + recall).",
        "ground_truth_url_keywords": ["model_evaluation"],
    },
    # --- 1 meta-вопрос (тест about.md) ---
    {
        "question": "Что ты умеешь?",
        "ground_truth": "Отвечаю на вопросы по трём разделам scikit-learn: линейные "
                        "модели, деревья решений, метрики качества.",
        "ground_truth_url_keywords": ["about.md", "local"],
    },
]

In [9]:
# Прямое подключение к RAG-pipeline без HTTP-слоя

from app.config import settings
from app.rag.chain import build_rag_chain
chain, retriever = build_rag_chain()

c:\Users\belon\anaconda3\envs\sklearn-rag\Lib\site-packages\qdrant_client\qdrant_remote.py:282: UserWarning: Qdrant client version 1.19.0 is incompatible with server version 1.12.0. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1106.89it/s]


In [10]:
# Recall@k для retriever'a

def url_match(returned_urls: list[str], expected_keywords: list[str]) -> bool:
    """Помечает retrieval как hit, если хоть один URL содержит ожидаемое ключевое слово.
    
    Используется в Recall@k: keyword-matching на уровне модуля sklearn
    (`linear_model` / `tree` / `model_evaluation`) — этого достаточно
    для понимания «не промазал ли retriever мимо темы целиком».
    """
    if not expected_keywords:
        return not returned_urls
    return any(
        any(kw.lower() in url.lower() for kw in expected_keywords)
        for url in returned_urls
    )

results = []
for item in GOLDEN:
    docs = retriever.invoke(item["question"])
    urls = [doc.metadata.get("source", "") for doc in docs]
    results.append({
        "question": item["question"],
        "ground_truth": item["ground_truth"],
        "retrieved_urls": urls,
        "retrieved_contexts": [doc.page_content for doc in docs],
        "retriever_hit": url_match(urls, item["ground_truth_url_keywords"]),
    })

hits = sum(1 for r in results if r["retriever_hit"])
recall_at_k = hits / len(results)
print(f"Retriever Recall@{settings.top_k}: {recall_at_k:.3f}  ({hits}/{len(results)})")

Retriever Recall@4: 1.000  (10/10)


In [ ]:
# Generation-метрики через RAGAS с сохранением прогресса
import hashlib
import time

from datasets import Dataset
from openai import RateLimitError
from ragas import evaluate
from ragas.metrics import Faithfulness, ResponseRelevancy
from ragas.metrics._faithfulness import NLIStatementInput, NLIStatementOutput
from ragas.run_config import RunConfig
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.rate_limiters import InMemoryRateLimiter

from app.llm import get_llm


def retry_minute_limit(fn):
    for attempt in range(5):
        try:
            return fn()
        except RateLimitError as exc:
            if "Request too large" in str(exc) or "per minute" not in str(exc) or attempt == 4:
                raise
            print("Минутный лимит Groq: жду 65 секунд перед повтором...")
            time.sleep(65)


notebook_dir = Path("notebooks") if Path("notebooks").is_dir() else Path(".")
progress_path = notebook_dir / "rag_eval_progress.json"
progress = json.loads(progress_path.read_text(encoding="utf-8")) if progress_path.exists() else globals().get("progress", {})


def save_progress():
    progress_path.write_text(json.dumps(progress, ensure_ascii=False, indent=2), encoding="utf-8")


# Ограничение нужно только для LLM-судьи: Groq допускает не более 1000 OTPM.
judge_llm = get_llm()
judge_llm.max_tokens = 900
judge_llm.rate_limiter = InMemoryRateLimiter(requests_per_second=1 / 20)
llm = LangchainLLMWrapper(judge_llm)
emb = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(
        model_name="intfloat/multilingual-e5-small",
        encode_kwargs={"normalize_embeddings": True},
    )
)
class SmallBatchFaithfulness(Faithfulness):
    async def _create_verdicts(self, row, statements, callbacks):
        context = "\n".join(row["retrieved_contexts"])
        verdicts = []
        for statement in statements:
            result = await self.nli_statements_prompt.generate(
                data=NLIStatementInput(context=context, statements=[statement]),
                llm=self.llm,
                callbacks=callbacks,
            )
            verdicts.extend(result.statements)
        return NLIStatementOutput(statements=verdicts)


metrics_to_run = [
    SmallBatchFaithfulness(llm=llm),
    ResponseRelevancy(llm=llm, embeddings=emb, strictness=1),
]
score_rows = []
for index, r in enumerate(results, 1):
    question = r["question"]
    fingerprint = hashlib.sha256(json.dumps({
        "model": settings.llm_model,
        "top_k": settings.top_k,
        "question": question,
        "retrieved_contexts": r["retrieved_contexts"],
        "strictness": 1,
    }, ensure_ascii=False).encode("utf-8")).hexdigest()
    entry = progress.get(question, {})
    if entry.get("fingerprint") != fingerprint:
        entry = {"fingerprint": fingerprint}
        progress[question] = entry

    if "response" not in entry:
        entry["response"] = r.get("response") or retry_minute_limit(
            lambda: chain.invoke(question)
        )
        save_progress()
    r["response"] = entry["response"]

    if "faithfulness" not in entry or "answer_relevancy" not in entry:
        row = Dataset.from_list([{
            "user_input": question,
            "response": r["response"],
            "retrieved_contexts": r["retrieved_contexts"],
            "reference": r["ground_truth"],
        }])
        scores = retry_minute_limit(lambda: evaluate(
            dataset=row,
            metrics=metrics_to_run,
            run_config=RunConfig(max_workers=1, max_retries=1, timeout=1200),
            raise_exceptions=True,
            show_progress=False,
        ))
        score = scores.to_pandas().iloc[0]
        entry["faithfulness"] = float(score["faithfulness"])
        entry["answer_relevancy"] = float(score["answer_relevancy"])
        save_progress()

    score_rows.append({
        "question": question,
        "faithfulness": entry["faithfulness"],
        "answer_relevancy": entry["answer_relevancy"],
        "evaluation_method": entry.get("evaluation_method", "ragas"),
    })
    print(f"Оценено {index}/{len(results)}: {question}")


In [14]:
# Сохранение сводки метрик в JSON

import pandas as pd

df = pd.DataFrame(score_rows)
gen_scores = {
    "faithfulness": float(df["faithfulness"].mean()),
    "answer_relevancy": float(df["answer_relevancy"].mean()),
}

metrics = {
    "n_questions": len(GOLDEN),
    "model": settings.llm_model,
    "embedding_model": settings.embedding_model,
    "top_k": settings.top_k,
    "retriever": {
        f"recall_at_{settings.top_k}": recall_at_k,
        "hits": hits,
    },
    "generation": gen_scores,
    "questions": score_rows,
}

(notebook_dir / "rag_metrics.json").write_text(
    json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(json.dumps(metrics, indent=2, ensure_ascii=False))

{
  "n_questions": 10,
  "model": "qwen/qwen3.8-27b",
  "embedding_model": "intfloat/multilingual-e5-small",
  "top_k": 4,
  "retriever": {
    "recall_at_4": 1.0,
    "hits": 10
  },
  "generation": {
    "faithfulness": 0.9535024154589371,
    "answer_relevancy": 0.737992711973908
  },
  "questions": [
    {
      "faithfulness": 1.0,
      "answer_relevancy": 0.9904948978418382
    },
    {
      "faithfulness": 0.875,
      "answer_relevancy": 0.9799872206516107
    },
    {
      "faithfulness": 0.9565217391304348,
      "answer_relevancy": 0.9594624874824469
    },
    {
      "faithfulness": 1.0,
      "answer_relevancy": 0.9511953929476531
    },
    {
      "faithfulness": 1.0,
      "answer_relevancy": 0.9469249551774093
    },
    {
      "faithfulness": 1.0,
      "answer_relevancy": 0.9469272534036879
    },
    {
      "faithfulness": 1.0,
      "answer_relevancy": 0.0
    },
    {
      "faithfulness": 0.75,
      "answer_relevancy": 0.0
    },
    {
      "faithfulness"